# Pipeline d'entraînement — California Housing

Ce notebook définit et soumet un pipeline complet avec Azure ML SDK v2 :
1. Préparation des données
2. Entraînement du modèle
3. Évaluation
4. Enregistrement conditionnel du modèle

In [ ]:
from azure.ai.ml import MLClient, command, Input, Output
from azure.ai.ml.dsl import pipeline
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Connecté au workspace :", ml_client.workspace_name)

In [ ]:
# Environnement réutilisé (déjà utilisé lors du test du cluster)
ENVIRONMENT = "azureml://registries/azureml/environments/sklearn-1.5/labels/latest"
COMPUTE = "cpu-cluster-train"  # ton cluster déjà configuré

In [ ]:
# --- Définition des composants du pipeline ---

prep_component = command(
    name="prep_data",
    display_name="Préparation des données",
    inputs={"input_data": Input(type="uri_folder")},
    outputs={
        "train_data": Output(type="uri_folder"),
        "val_data": Output(type="uri_folder"),
    },
    code="./prep_src",
    command=(
        "python prep.py "
        "--input_data ${{inputs.input_data}} "
        "--train_data ${{outputs.train_data}} "
        "--val_data ${{outputs.val_data}}"
    ),
    environment=ENVIRONMENT,
    compute=COMPUTE,
)

# Composant d'entraînement PREPARE POUR LE SWEEP :
# entraîne ET évalue en un seul essai, logge le R2 comme métrique cible.
train_sweep_component = command(
    name="train_sweep_trial",
    display_name="Entraînement (essai du sweep)",
    inputs={
        "train_data": Input(type="uri_folder"),
        "val_data": Input(type="uri_folder"),
        "n_estimators": Input(type="integer", default=200),
        "max_depth": Input(type="integer", default=15),
    },
    outputs={"model_output": Output(type="uri_folder")},
    code="./train_src",
    command=(
        "python train_sweep.py "
        "--train_data ${{inputs.train_data}} "
        "--val_data ${{inputs.val_data}} "
        "--n_estimators ${{inputs.n_estimators}} "
        "--max_depth ${{inputs.max_depth}} "
        "--model_output ${{outputs.model_output}}"
    ),
    environment=ENVIRONMENT,
    compute=COMPUTE,
)

evaluate_component = command(
    name="evaluate_model",
    display_name="Évaluation du modèle",
    inputs={
        "val_data": Input(type="uri_folder"),
        "model_input": Input(type="uri_folder"),
    },
    outputs={"metrics_output": Output(type="uri_folder")},
    code="./evaluate_src",
    command=(
        "python evaluate.py "
        "--val_data ${{inputs.val_data}} "
        "--model_input ${{inputs.model_input}} "
        "--metrics_output ${{outputs.metrics_output}}"
    ),
    environment=ENVIRONMENT,
    compute=COMPUTE,
)

register_component = command(
    name="register_model",
    display_name="Enregistrement du modèle",
    inputs={
        "model_input": Input(type="uri_folder"),
        "metrics_input": Input(type="uri_folder"),
        "model_name": Input(type="string", default="california-housing-model"),
        "r2_threshold": Input(type="number", default=0.7),
    },
    code="./register_src",
    command=(
        "python register.py "
        "--model_input ${{inputs.model_input}} "
        "--metrics_input ${{inputs.metrics_input}} "
        "--model_name ${{inputs.model_name}} "
        "--r2_threshold ${{inputs.r2_threshold}}"
    ),
    environment=ENVIRONMENT,
    compute=COMPUTE,
)

In [ ]:
# --- Assemblage du pipeline complet (avec sweep d'hyperparamètres) ---

from azure.ai.ml.sweep import Choice, BanditPolicy


@pipeline(
    name="pipeline_entrainement_california_housing",
    display_name="Pipeline entraînement - California Housing",
    compute=COMPUTE,
)
def training_pipeline(input_data, r2_threshold=0.7):
    prep_step = prep_component(input_data=input_data)

    # --- Étape de sweep : plusieurs essais avec des hyperparamètres différents ---
    train_step = train_sweep_component(
        train_data=prep_step.outputs.train_data,
        val_data=prep_step.outputs.val_data,
        n_estimators=Choice([100, 200, 300, 400]),
        max_depth=Choice([5, 10, 15, 20, 25]),
    )
    sweep_step = train_step.sweep(
        primary_metric="r2_score",
        goal="Maximize",
        sampling_algorithm="random",
    )
    # Limite adaptée au quota du cluster (max 1 nœud => essais séquentiels)
    sweep_step.set_limits(max_total_trials=8, max_concurrent_trials=1, timeout=3600)
    sweep_step.early_termination = BanditPolicy(
        slack_factor=0.1, evaluation_interval=1, delay_evaluation=2
    )

    # --- Évaluation finale du meilleur modèle trouvé par le sweep ---
    evaluate_step = evaluate_component(
        val_data=prep_step.outputs.val_data,
        model_input=sweep_step.outputs.model_output,
    )

    # --- Enregistrement conditionnel ---
    register_component(
        model_input=sweep_step.outputs.model_output,
        metrics_input=evaluate_step.outputs.metrics_output,
        r2_threshold=r2_threshold,
    )

    return {}

In [ ]:
# --- Récupération du Data Asset versionné (Version 1) créé précédemment ---

raw_data_asset = ml_client.data.get(name="california-housing-raw", version="1")
print("Data Asset utilisé :", raw_data_asset.path)

pipeline_job = training_pipeline(
    input_data=Input(type="uri_folder", path=raw_data_asset.id),
    r2_threshold=0.7,
)

In [ ]:
# --- Soumission du pipeline ---

submitted_job = ml_client.jobs.create_or_update(
    pipeline_job,
    experiment_name="pipeline-california-housing",
)
print("Job soumis, suivre ici :", submitted_job.studio_url)